In [13]:
import pandas as pd
import re
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
import pickle

# ── Tạo lại sessions từ file gốc ──────────────────────────
print("Đang đọc và parse log (mất vài phút)...")
pattern = re.compile(r'(blk_-?\d+)')
records = []

with open('../data/HDFS.log', 'r', encoding='utf-8') as f:
    for line in f:
        for blk in set(pattern.findall(line)):
            records.append({'BlockId': blk, 'Content': line.strip()})

df_raw   = pd.DataFrame(records)
sessions = df_raw.groupby('BlockId')['Content'].apply(list).reset_index()
labels   = pd.read_csv('../data/anomaly_label.csv')
sessions = sessions.merge(labels, on='BlockId')
print(f"✅ Tổng sessions: {len(sessions)}")

Đang đọc và parse log (mất vài phút)...
✅ Tổng sessions: 575061


In [14]:
sessions['text']      = sessions['Content'].apply(lambda x: ' '.join(x))
sessions['label_int'] = (sessions['Label'] == 'Anomaly').astype(int)

X = sessions['text'].values
y = sessions['label_int'].values

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

Train: 402542 | Val: 86259 | Test: 86260


In [15]:
vectorizer   = TfidfVectorizer(max_features=500)
X_train_vec  = vectorizer.fit_transform(X_train).toarray()
X_val_vec    = vectorizer.transform(X_val).toarray()
X_test_vec   = vectorizer.transform(X_test).toarray()

print(f"Shape train: {X_train_vec.shape}")  # (402543, 500)

Shape train: (402542, 500)


In [16]:
os.makedirs('../data/processed', exist_ok=True)

np.save('../data/processed/X_train_vec.npy', X_train_vec)
np.save('../data/processed/X_val_vec.npy',   X_val_vec)
np.save('../data/processed/X_test_vec.npy',  X_test_vec)
np.save('../data/processed/y_train.npy',     y_train)
np.save('../data/processed/y_val.npy',       y_val)
np.save('../data/processed/y_test.npy',      y_test)

with open('../data/processed/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

print("✅ Đã lưu 6 file .npy và vectorizer.pkl vào data/processed/")

✅ Đã lưu 6 file .npy và vectorizer.pkl vào data/processed/
